# RoadVision-AI Training Notebook
This notebook trains YOLOv8 models for pothole detection and stair detection. It covers setting up the environment, getting data, training, evaluation, and exporting to ONNX.

In [ ]:
# 1. Check GPU and Mount Google Drive
import torch

if torch.cuda.is_available():
    print(f"✅ GPU is enabled: {torch.cuda.get_device_name(0)}")
else:
    print("❌ GPU is NOT enabled. Please go to Runtime -> Change runtime type -> select T4 GPU")

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Install Dependencies
!pip install ultralytics
import ultralytics
ultralytics.checks()

In [ ]:
# 3. Clone Repository and Unzip Datasets
import os

WORK_DIR = '/content/RoadVision-AI'

if not os.path.exists(WORK_DIR):
    !git clone https://github.com/mahdi-misk/RoadVision-AI.git {WORK_DIR}

%cd {WORK_DIR}

# Unzip Data from Google Drive
# Make sure you uploaded 'potholes_processed.zip' and 'stairs_processed.zip'
# inside a folder named 'RoadVision-Data' in your Google Drive.
!mkdir -p datasets/potholes/processed
!mkdir -p datasets/stairs/processed

!unzip -q /content/drive/MyDrive/RoadVision-Data/potholes_processed.zip -d datasets/potholes/processed
!unzip -q /content/drive/MyDrive/RoadVision-Data/stairs_processed.zip -d datasets/stairs/processed

print("✅ Project and Data are ready!")

In [ ]:
# 4. Train Pothole Model
!yolo detect train model=yolov8n.pt data=datasets/potholes/processed/data.yaml epochs=80 imgsz=640 batch=16 project=runs/pothole name=pothole_yolov8_final

In [ ]:
# 5. Train Stairs Model
!yolo detect train model=yolov8n.pt data=datasets/stairs/processed/data.yaml epochs=80 imgsz=640 batch=16 project=runs/stairs name=stairs_yolov8_final

In [ ]:
# 6. Validate & Predict (Potholes)
!yolo detect val model=runs/pothole/pothole_yolov8_final/weights/best.pt data=datasets/potholes/processed/data.yaml split=test project=runs/pothole name=val_results
!yolo detect predict model=runs/pothole/pothole_yolov8_final/weights/best.pt source=datasets/potholes/processed/images/test project=runs/pothole name=predict_results save=True

In [ ]:
# 7. Validate & Predict (Stairs)
!yolo detect val model=runs/stairs/stairs_yolov8_final/weights/best.pt data=datasets/stairs/processed/data.yaml split=test project=runs/stairs name=val_results
!yolo detect predict model=runs/stairs/stairs_yolov8_final/weights/best.pt source=datasets/stairs/processed/images/test project=runs/stairs name=predict_results save=True

In [ ]:
# 8. Export Models to ONNX
!yolo export model=runs/pothole/pothole_yolov8_final/weights/best.pt format=onnx
!yolo export model=runs/stairs/stairs_yolov8_final/weights/best.pt format=onnx

In [ ]:
# 9. Save Models and Results to Google Drive
!mkdir -p /content/drive/MyDrive/RoadVision-Data/models/potholes
!mkdir -p /content/drive/MyDrive/RoadVision-Data/models/stairs

# Copy Pothole best.pt and best.onnx
!cp runs/pothole/pothole_yolov8_final/weights/best.pt /content/drive/MyDrive/RoadVision-Data/models/potholes/pothole_yolov8_final.pt
!cp runs/pothole/pothole_yolov8_final/weights/best.onnx /content/drive/MyDrive/RoadVision-Data/models/potholes/pothole_yolov8_final.onnx

# Copy Stairs best.pt and best.onnx
!cp runs/stairs/stairs_yolov8_final/weights/best.pt /content/drive/MyDrive/RoadVision-Data/models/stairs/stairs_yolov8_final.pt
!cp runs/stairs/stairs_yolov8_final/weights/best.onnx /content/drive/MyDrive/RoadVision-Data/models/stairs/stairs_yolov8_final.onnx

print("✅ Training finished and models saved to Google Drive!")